# Agentic RAG with LangGraph

**Agentic RAG** goes beyond a fixed `retrieve → generate` chain: the system **grades** context, **rewrites** weak queries, **loops** until evidence is sufficient, or lets an **agent** call **`search_docs`** dynamically. LangGraph unifies both styles — explicit **graph RAG** (**05**) and **tool-loop agents** (**06**) — behind a **hybrid router**.

This notebook builds: mock/Chroma-style retriever, **grade** node (`relevant` / `irrelevant`), rewrite loop, grounded **generate**, a **search_docs** ReAct agent with **structured citations**, hybrid routing, and comparison to **01. LangChain/11** LCEL RAG.

Prerequisites: **05. Building Your First Graph**, **06. Agent Loops and the ReAct Pattern**, **01. LangChain/11** (LCEL RAG).

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json

import langchain_core

try:
    import langgraph
    print("langgraph:", langgraph.__version__)
except ImportError:
    print("langgraph: pip install langgraph")

print("langchain-core:", langchain_core.__version__)

---

## 1. Agentic RAG — Two Control Strategies

```
HYBRID ROUTER (this notebook)
         │
    ┌────┴────┐
    ▼         ▼
GRAPH PATH   AGENT PATH
(fixed)      (tool loop)
    │              │
retrieve         search_docs
    │              │ tool
 grade            model ◄──┐
    │                        │
 rewrite loop                 │
    │                        │
 generate ──► answer    final answer + citations
```

| Path | When to use |
|------|-------------|
| **Graph RAG** | Predictable latency, auditable stages, grade/rewrite |
| **Agent RAG** | Multi-hop retrieval, model decides when to search |
| **Hybrid** | Simple factual → graph; exploratory → agent |

---

## 2. Unified State Schema

One state object serves **both** paths — graph keys and agent **`messages`**:

In [ ]:
import operator
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langgraph.graph.message import add_messages


class AgenticRAGState(TypedDict):
    question: str
    context: str
    messages: Annotated[list, add_messages]
    source_ids: list[str]
    grade: str
    answer: str
    route: str              # graph | agent
    attempts: int
    citations: list[dict]   # structured [{"id": "c2", "snippet": "..."}]
    trace: Annotated[list[str], operator.add]

| Key | Merge | Used by |
|-----|-------|--------|
| `question` | replace | Both paths; rewritten on retry |
| `context` | replace | Graph path retrieval |
| `messages` | `add_messages` | Agent path ReAct loop |
| `source_ids` | replace | Citations from retriever |
| `grade` | replace | `relevant` / `irrelevant` |
| `route` | replace | Hybrid router decision |
| `trace` | append | Debug node path |

---

## 3. Teaching Corpus and Retriever

Same five chunks as **05** / **01. LangChain/11**. **`ChromaRetriever`** wraps keyword search — swap the inner call for real Chroma in production:

In [ ]:
CORPUS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI. Routes live under /notes with GET, POST, PUT, DELETE."},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling revisions."},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Send Authorization: Bearer token header."},
    {"id": "c4", "text": "Pytest fixtures share database setup. Use conftest.py for session-scoped engines."},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live schema and drafts revision files."},
]

DOCS_KEYWORDS = {"jwt", "alembic", "pytest", "fastapi", "migration", "token", "route", "conftest"}


def keyword_retrieve(query: str, k: int = 2) -> tuple[str, list[str], list[dict]]:
    """In-memory mock retriever — same scoring as 05."""
    tokens = set(query.lower().split())
    scored = []
    for doc in CORPUS:
        doc_tokens = set(doc["text"].lower().split())
        score = len(tokens & doc_tokens)
        scored.append((score, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    top = [d for s, d in scored if s > 0][:k] or [CORPUS[0]]
    context = "\n".join(f"[{d['id']}] {d['text']}" for d in top)
    ids = [d["id"] for d in top]
    citations = [{"id": d["id"], "snippet": d["text"][:80]} for d in top]
    return context, ids, citations


class ChromaRetriever:
    """Teaching stub — replace .search() body with chromadb collection query."""

    def __init__(self, collection_name: str = "notes_api"):
        self.collection_name = collection_name

    def search(self, query: str, k: int = 2) -> tuple[str, list[str], list[dict]]:
        # Production: vectorstore.similarity_search(query, k=k)
        return keyword_retrieve(query, k=k)


retriever = ChromaRetriever()
ctx, ids, cites = retriever.search("alembic upgrade")
print("context preview:", ctx[:120])
print("source_ids:", ids)

---

## 4. Graph RAG Nodes — Retrieve, Grade, Rewrite, Generate

Ported from **05** with citation extraction:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)


def retrieve_node(state: AgenticRAGState) -> dict:
    context, ids, citations = retriever.search(state["question"])
    return {
        "context": context,
        "source_ids": ids,
        "citations": citations,
        "attempts": state.get("attempts", 0) + 1,
        "trace": ["retrieve"],
    }


def grade_node(state: AgenticRAGState) -> dict:
    q = state["question"].lower()
    ctx = state.get("context", "").lower()
    relevant = any(k in ctx for k in q.split() if len(k) > 3) or any(
        k in q and k in ctx for k in DOCS_KEYWORDS
    )
    grade = "relevant" if relevant else "irrelevant"
    return {"grade": grade, "trace": ["grade"]}


def rewrite_node(state: AgenticRAGState) -> dict:
    expanded = state["question"] + " (FastAPI Notes API alembic jwt pytest)"
    return {"question": expanded, "trace": ["rewrite"]}


rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer using ONLY the context. Cite chunk ids in brackets e.g. [c2].
If insufficient, say so briefly."""),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])
generate_chain = rag_prompt | llm


def generate_node(state: AgenticRAGState) -> dict:
    msg = generate_chain.invoke({"context": state["context"], "question": state["question"]})
    return {"answer": msg.content, "trace": ["generate"]}

**Upgrade path:** replace keyword **`grade_node`** with an LLM structured grader (`relevant: bool`) for production.

---

## 5. Graph Routers — Grade and Rewrite Loop

Thin routers from **04** / **05**:

In [ ]:
from langgraph.graph import END

MAX_ATTEMPTS = 2


def route_grade(state: AgenticRAGState) -> Literal["relevant", "irrelevant"]:
    return "relevant" if state["grade"] == "relevant" else "irrelevant"


def route_rewrite(state: AgenticRAGState):
    return "retrieve" if state.get("attempts", 0) < MAX_ATTEMPTS else END

---

## 6. Build Graph RAG Subgraph

Extract the fixed pipeline as a compilable subgraph (**10**):

In [ ]:
from langgraph.graph import START, StateGraph


def build_graph_rag_subgraph():
    builder = StateGraph(AgenticRAGState)
    builder.add_node("retrieve", retrieve_node)
    builder.add_node("grade", grade_node)
    builder.add_node("rewrite", rewrite_node)
    builder.add_node("generate", generate_node)

    builder.add_edge(START, "retrieve")
    builder.add_edge("retrieve", "grade")
    builder.add_conditional_edges(
        "grade", route_grade, {"relevant": "generate", "irrelevant": "rewrite"}
    )
    builder.add_conditional_edges("rewrite", route_rewrite)
    builder.add_edge("generate", END)
    return builder.compile()


graph_rag = build_graph_rag_subgraph()
print("graph RAG nodes:", list(graph_rag.get_graph().nodes))

---

## 7. Agent Path — `search_docs` Tool + ReAct Loop

The agent **decides** when to search — multi-hop friendly (**06**):

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition


@tool
def search_docs(query: str) -> str:
    """Search Notes API documentation. Returns chunks with [c#] ids."""
    context, ids, _ = retriever.search(query)
    return context or "No results."


AGENT_SYSTEM = """You are a Notes API documentation agent.
Use search_docs before answering factual questions.
Always cite chunk ids like [c3] in your final answer."""

agent_tools = [search_docs]
llm_with_tools = llm.bind_tools(agent_tools)


class AgentOnlyState(TypedDict):
    messages: Annotated[list, add_messages]
    source_ids: list[str]
    citations: list[dict]
    answer: str
    trace: Annotated[list[str], operator.add]


def agent_model_node(state: AgentOnlyState) -> dict:
    msgs = [{"role": "system", "content": AGENT_SYSTEM}] + state["messages"]
    return {"messages": [llm_with_tools.invoke(msgs)], "trace": ["agent_model"]}


def extract_citations_node(state: AgentOnlyState) -> dict:
    """Parse [c#] tags from final answer into structured citations."""
    last = ""
    for msg in reversed(state["messages"]):
        if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
            last = msg.content if isinstance(msg.content, str) else str(msg.content)
            break
    ids = []
    citations = []
    for doc in CORPUS:
        tag = f"[{doc['id']}]"
        if tag in last:
            ids.append(doc["id"])
            citations.append({"id": doc["id"], "snippet": doc["text"][:80]})
    return {"answer": last, "source_ids": ids, "citations": citations, "trace": ["extract_citations"]}


def agent_should_continue(state: AgentOnlyState):
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "agent_tools"
    return "extract_citations"


def build_agent_rag_subgraph():
    builder = StateGraph(AgentOnlyState)
    builder.add_node("agent_model", agent_model_node)
    builder.add_node("agent_tools", ToolNode(agent_tools))
    builder.add_node("extract_citations", extract_citations_node)
    builder.add_edge(START, "agent_model")
    builder.add_conditional_edges("agent_model", agent_should_continue)
    builder.add_edge("agent_tools", "agent_model")
    builder.add_edge("extract_citations", END)
    return builder.compile()


agent_rag = build_agent_rag_subgraph()
print("agent RAG nodes:", list(agent_rag.get_graph().nodes))

---

## 8. Hybrid Router — Graph vs Agent Path

Route **straightforward factual** questions to the graph; **multi-part / exploratory** to the agent:

In [ ]:
def classify_route(question: str) -> Literal["graph", "agent"]:
    q = question.lower()
    if any(w in q for w in ("compare", "and also", "step by step", "multiple", "both")):
        return "agent"
    if any(w in q for w in DOCS_KEYWORDS):
        return "graph"
    return "agent"


def hybrid_router_node(state: AgenticRAGState) -> dict:
    route = classify_route(state["question"])
    return {"route": route, "trace": [f"router:{route}"]}


def route_hybrid(state: AgenticRAGState) -> Literal["graph", "agent"]:
    return state["route"]  # type: ignore[return-value]

---

## 9. Hybrid Graph Factory — `build_agentic_rag_graph()`

Parent graph wires router + both subgraph invocations:

In [ ]:
def graph_path_node(state: AgenticRAGState) -> dict:
    sub = graph_rag.invoke(state)
    return {
        "answer": sub.get("answer", ""),
        "context": sub.get("context", ""),
        "source_ids": sub.get("source_ids", []),
        "citations": sub.get("citations", []),
        "grade": sub.get("grade", ""),
        "trace": sub.get("trace", []),
    }


def agent_path_node(state: AgenticRAGState) -> dict:
    sub = agent_rag.invoke({
        "messages": [HumanMessage(content=state["question"])],
        "source_ids": [],
        "citations": [],
        "answer": "",
        "trace": [],
    })
    return {
        "answer": sub.get("answer", ""),
        "source_ids": sub.get("source_ids", []),
        "citations": sub.get("citations", []),
        "messages": sub.get("messages", []),
        "trace": sub.get("trace", []),
    }


def build_agentic_rag_graph():
    builder = StateGraph(AgenticRAGState)
    builder.add_node("router", hybrid_router_node)
    builder.add_node("graph_path", graph_path_node)
    builder.add_node("agent_path", agent_path_node)

    builder.add_edge(START, "router")
    builder.add_conditional_edges(
        "router", route_hybrid, {"graph": "graph_path", "agent": "agent_path"}
    )
    builder.add_edge("graph_path", END)
    builder.add_edge("agent_path", END)
    return builder.compile()


agentic_rag = build_agentic_rag_graph()
print("hybrid nodes:", list(agentic_rag.get_graph().nodes))

---

## 10. Invoke — Graph Path (Retrieve → Grade → Generate)

Direct subgraph smoke test:

In [ ]:
GRAPH_INITIAL = {
    "context": "", "messages": [], "source_ids": [], "grade": "",
    "answer": "", "route": "graph", "attempts": 0, "citations": [], "trace": [],
}

g_out = graph_rag.invoke({**GRAPH_INITIAL, "question": "What Alembic command applies migrations?"})
print("trace:", g_out["trace"])
print("grade:", g_out["grade"])
print("sources:", g_out["source_ids"])
print("answer:", g_out["answer"][:200])

---

## 11. Invoke — Agent Path (Tool Loop)

Agent may call **`search_docs`** multiple times before answering:

In [ ]:
a_out = agent_rag.invoke({
    "messages": [HumanMessage(content="JWT header format and where pytest fixtures live — cite both.")],
    "source_ids": [], "citations": [], "answer": "", "trace": [],
})
print("trace:", a_out["trace"])
print("citations:", a_out["citations"])
print("answer:", a_out["answer"][:250])

---

## 12. Structured Citations Response

Wrap hybrid output for a stable API contract (**16**):

In [ ]:
HYBRID_INITIAL = {
    "context": "", "messages": [], "source_ids": [], "grade": "",
    "answer": "", "route": "", "attempts": 0, "citations": [], "trace": [],
}


def run_agentic_rag(question: str) -> dict:
    state = agentic_rag.invoke({**HYBRID_INITIAL, "question": question})
    return {
        "question": question,
        "answer": state["answer"],
        "route": state["route"],
        "source_ids": state.get("source_ids", []),
        "citations": state.get("citations", []),
        "path": state.get("trace", []),
        "grade": state.get("grade", ""),
    }


for q in [
    "Alembic upgrade command?",
    "Compare JWT auth and pytest fixtures — cite both chunks.",
    "Kubernetes ingress controller",
]:
    result = run_agentic_rag(q)
    print(json.dumps(result, indent=2)[:500], "...\n")

---

## 13. Rewrite Loop in Action

Weak queries trigger **rewrite → retrieve** until **`MAX_ATTEMPTS`**:

In [ ]:
weak = graph_rag.invoke({**GRAPH_INITIAL, "question": "database schema versioning tool"})
print("attempts:", weak["attempts"])
print("trace:", weak["trace"])
print("rewritten question contains alembic:", "alembic" in weak["question"].lower())

---

## 14. LangGraph Agentic RAG vs LCEL RAG (**01. LangChain/11**)

| Capability | **LCEL RAG chain** | **This agentic graph** |
|------------|---------------------|------------------------|
| Linear retrieve → generate | ✅ `retriever \| prompt \| llm` | ✅ `graph_path` subgraph |
| Grade irrelevant docs | Manual branch | ✅ **`grade_node`** + conditional |
| Query rewrite loop | Awkward | ✅ **`rewrite → retrieve`** cycle |
| Agent-chosen retrieval | Not native | ✅ **`agent_path`** ReAct loop |
| Hybrid routing | Multiple chains | ✅ Single **`build_agentic_rag_graph()`** |
| Structured citations | Custom parser | ✅ **`citations`** in state |
| Checkpoint / HITL | Manual | ✅ `compile(checkpointer=...)` (**08**, **09**) |

Production pattern: keep LCEL **`generate_chain`** **inside** `generate_node`; LangGraph owns **control flow** around it.

---

## 15. Swapping In Real Chroma

Replace the mock body of **`ChromaRetriever.search`** — graph and agent code stay unchanged:

In [ ]:
# Uncomment when chromadb is installed:
# from langchain_chroma import Chroma
# from langchain_openai import OpenAIEmbeddings
#
# vectorstore = Chroma.from_texts(
#     texts=[d["text"] for d in CORPUS],
#     metadatas=[{"id": d["id"]} for d in CORPUS],
#     embedding=OpenAIEmbeddings(api_key=OPENAI_API_KEY),
#     collection_name="notes_api",
# )
#
# def chroma_search(query: str, k: int = 2):
#     docs = vectorstore.similarity_search(query, k=k)
#     context = "\n".join(f"[{d.metadata['id']}] {d.page_content}" for d in docs)
#     ids = [d.metadata["id"] for d in docs]
#     citations = [{"id": d.metadata["id"], "snippet": d.page_content[:80]} for d in docs]
#     return context, ids, citations

print("Chroma swap: replace ChromaRetriever.search() internals only")

---

## 16. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| No grade before generate | Hallucination on bad retrieval | Always **`grade_node`** |
| Uncapped rewrite loop | Infinite retrieve cycles | `MAX_ATTEMPTS` in router |
| Agent path without citations | Unverifiable answers | **`extract_citations_node`** |
| Same retriever in graph + tool | Drift between paths | Single **`retriever`** instance |
| Mega-node does retrieve+grade+gen | Untestable | Split nodes like **05** |
| Wrong agent conditional edges | Skips tool loop | `tools_condition` or custom router |
| Ignoring `source_ids` in response | Broken UI citations | Return structured **`citations`** |

---

## 17. Summary

You built **agentic RAG** combining **05**'s retrieve-grade-rewrite-generate graph with **06**'s **`search_docs`** ReAct agent, unified under **`AgenticRAGState`** and a **hybrid router** in **`build_agentic_rag_graph()`**.

Key takeaways:

- **Graph path** = predictable stages with **grade** branch and **rewrite** loop.
- **Agent path** = model-driven retrieval via **`search_docs`** tool loop.
- **State** carries `question`, `context`, `messages`, `source_ids`, `grade`, and **`citations`**.
- **`ChromaRetriever`** stub swaps to real Chroma without graph changes.
- **Hybrid routing** picks graph vs agent by query shape.
- LangGraph adds **loops and branches** LCEL RAG chains handle awkwardly (**01. LangChain/11**).

Next: **13. Streaming, Batch, and Async** — stream node updates and agent tokens from these graphs.